In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/ETHZ_ALL"

import numpy as np
import pandas as pd
import glob
import os
import gc
import logging
from typing import Generator, List, Set
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
import pytz


import pytz

def detect_ev_probability_pv_aware(
    customer_df: pd.DataFrame,
    step_h: float = 0.25,
    max_yearly_mwh: float = 100.0
):
    """
    PV-aware, grid-aware EV detection with soft probability score.
    Returns:
      - result dict (used in final table)
      - sessions DataFrame (optional, kept for auditability)
    """

    df = customer_df.copy()

    # ------------------------------------------------------------------
    # Timezone handling (keep UTC internally)
    # ------------------------------------------------------------------
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")

    # ------------------------------------------------------------------
    # Rename to match earlier logic
    # CONSO_KWH / PROD_KWH are energy per 15 min
    # Convert to kW for detection logic
    # ------------------------------------------------------------------
    df["Consommation"] = df["CONSO_KWH"] / step_h
    df["Excedent"] = df["PROD_KWH"] / step_h

    # ------------------------------------------------------------------
    # 1. Yearly consumption filter
    # ------------------------------------------------------------------
    yearly_mwh = df["CONSO_KWH"].sum() / 1000.0
    if yearly_mwh >= max_yearly_mwh:
        return {
            "EV_probability": 0.0,
            "filtered_out": True,
            "yearly_consumption_MWh": round(yearly_mwh, 2),
            "grid_sessions": 0,
            "pv_sessions": 0,
            "total_sessions": 0,
        }, pd.DataFrame()

    # ------------------------------------------------------------------
    # 2. Baseload estimation (night hours)
    # ------------------------------------------------------------------
    df["hour"] = df.index.hour
    df["net_load"] = df["Consommation"] - df["Excedent"]

    night_mask = (df["hour"] >= 1) & (df["hour"] < 4)
    baseload = df.loc[night_mask, "net_load"].median()

    # ------------------------------------------------------------------
    # 3. Session extraction helper
    # ------------------------------------------------------------------
    def extract_sessions(mask, load_col, min_len=6, max_rel_std=0.15):
        groups, cur = [], []
        for i, ok in enumerate(mask):
            if ok:
                cur.append(i)
            else:
                if len(cur) >= min_len:
                    groups.append(cur)
                cur = []
        if len(cur) >= min_len:
            groups.append(cur)

        sessions = []
        for g in groups:
            load = load_col.iloc[g]
            mean_p = load.mean()
            if mean_p > 0 and load.std() / mean_p <= max_rel_std:
                duration_h = len(g) * step_h
                sessions.append({
                    "start": df.index[g[0]],
                    "duration_h": duration_h,
                    "mean_power_kW": mean_p,
                    "energy_kWh": mean_p * duration_h,
                })
        return pd.DataFrame(sessions)

    # ------------------------------------------------------------------
    # 4. Grid EV detection
    # ------------------------------------------------------------------
    df["active_load"] = df["net_load"] - baseload
    grid_mask = df["active_load"] >= 2.8
    grid_sessions = extract_sessions(grid_mask, df["active_load"])

    # ------------------------------------------------------------------
    # 5. PV-aware EV detection
    # ------------------------------------------------------------------
    pv_mask = (
        (df["Excedent"].diff() < -0.5) &
        (df["net_load"].rolling(4).std() < 0.25) &
        (df["net_load"] > 0.8)
    )
    pv_sessions = extract_sessions(pv_mask, df["net_load"])

    sessions = pd.concat([grid_sessions, pv_sessions], ignore_index=True)

    # ------------------------------------------------------------------
    # 6. Soft EV probability score
    # ------------------------------------------------------------------
    def clip01(x): return max(0.0, min(1.0, x))
    def logistic(x, x0, k): return 1 / (1 + np.exp(-k * (x - x0)))

    if sessions.empty:
        ev_probability = 0.0
    else:
        sessions["week"] = (
            sessions["start"]
            .dt.tz_localize(None)
            .dt.to_period("W")
        )

        n_sessions = len(sessions)
        max_weekly = sessions.groupby("week").size().max()
        mean_power = sessions["mean_power_kW"].mean()
        mean_energy = sessions["energy_kWh"].mean()
        flatness = sessions["energy_kWh"].std() / (mean_energy + 1e-6)

        ev_probability = clip01(
            0.30 * logistic(n_sessions, 8, 0.4) +
            0.25 * logistic(max_weekly, 2, 1.2) +
            0.20 * clip01((mean_power - 0.8) / (2.5 - 0.8)) +
            0.15 * clip01((mean_energy - 4) / (20 - 4)) +
            0.10 * clip01(1 - flatness)
        )

    # ------------------------------------------------------------------
    # 7. Output
    # ------------------------------------------------------------------
    result = {
        "EV_probability": round(ev_probability, 3),
        "filtered_out": False,
        "yearly_consumption_MWh": round(yearly_mwh, 2),
        "grid_sessions": len(grid_sessions),
        "pv_sessions": len(pv_sessions),
        "total_sessions": len(sessions),
    }

    return result, sessions


# --- Configure Logging --------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("data_loading.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# --- Worker function ----------------------------------------------------------
def process_single_file(file_path: str, valid_ids_set: Set[str], target_meta: pd.DataFrame) -> pd.DataFrame:
    try:
        df = pd.read_parquet(
            file_path,
            columns=["ID", "DT_UTC", "CONSO_KWH", "PROD_KWH"],
            engine='pyarrow'
        )

        if df.empty:
            return pd.DataFrame()

        df["ID"] = df["ID"].astype(str)
        filtered_df = df[df["ID"].isin(valid_ids_set)].copy()

        if filtered_df.empty:
            return pd.DataFrame()

        filtered_df["DT_UTC"] = pd.to_datetime(filtered_df["DT_UTC"])
        return filtered_df.merge(target_meta, on="ID", how="left")

    except Exception as e:
        logger.error(f"Failed to process {file_path}: {str(e)}")
        return pd.DataFrame()


# --- Parallel Loader ----------------------------------------------------------
def load_all_data_parallel_generator(
    data_dir: str,
    partner_type: str = "Particuliers",
    batch_size: int = 2
) -> Generator[pd.DataFrame, None, None]:

    logger.info(f"Starting data load for type: {partner_type}")

    # ✅ Load Metadata
    try:
        meta_df = pd.read_parquet(os.path.join(data_dir, "metadata"), engine='pyarrow')
        target_meta = meta_df[meta_df["TYPE_PARTENAIRE_LIBELLE"] == partner_type].copy()
        target_meta["ID"] = target_meta["ID"].astype(str)
        valid_ids_set = set(target_meta["ID"])
        logger.info(f"Metadata loaded. {len(valid_ids_set)} valid IDs.")
    except Exception as e:
        logger.critical(f"Could not load metadata: {e}")
        return

    # ✅ Find parquet files
    files = [f for f in glob.glob(os.path.join(data_dir, "*.parquet")) if "metadata" not in f.lower()]
    if not files:
        logger.error(f"No parquet files found in {data_dir}")
        return

    n_cores = max(1, cpu_count() // 2)
    batch_size = batch_size or n_cores

    worker_func = partial(process_single_file, valid_ids_set=valid_ids_set, target_meta=target_meta)

    with tqdm(total=len(files), desc="Overall Progress") as pbar:
        with Pool(processes=n_cores) as pool:
            for i in range(0, len(files), batch_size):
                file_chunk = files[i : i + batch_size]

                results = pool.map(worker_func, file_chunk)
                pbar.update(len(file_chunk))

                batch_df = pd.concat([res for res in results if not res.empty], ignore_index=True)
                if not batch_df.empty:
                    yield batch_df

                del results
                gc.collect()

    logger.info("✅ Data loading sequence complete.")


# =====================================================================
# ✅ MAIN EXECUTION BLOCK NEEDED FOR GOOGLE COLAB MULTIPROCESSING
# =====================================================================
if __name__ == "__main__":
    results = []

    data_gen = load_all_data_parallel_generator(
        data_dir=data_dir,
        partner_type="Particuliers",
        batch_size=4
    )

    for batch_df in data_gen:

        for cust_id, cust_df in batch_df.groupby("ID"):


            cust_df = cust_df.set_index("DT_UTC").sort_index()

            ev_result, sessions = detect_ev_probability_pv_aware(cust_df)

            results.append({
                "ID": cust_id,
                "EV_probability": ev_result["EV_probability"],
                "yearly_consumption_MWh": ev_result["yearly_consumption_MWh"],
                "grid_sessions": ev_result["grid_sessions"],
                "pv_sessions": ev_result["pv_sessions"],
                "total_sessions": ev_result["total_sessions"],
                "filtered_out": ev_result["filtered_out"],
                "avg_conso_kWh": cust_df["CONSO_KWH"].mean()
            })



        del batch_df
        gc.collect()
        print("✅ One batch done!")

    final = pd.DataFrame(results)

    output_path = "/content/drive/MyDrive/ev_detection_newresults4.parquet"
    final.to_parquet(output_path)

    print("✅ Processing complete! Results saved to:", output_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Overall Progress:   2%|▏         | 4/249 [00:05<05:26,  1.33s/it]

✅ One batch done!


Overall Progress:   3%|▎         | 8/249 [00:19<10:18,  2.57s/it]

✅ One batch done!


...

✅ One batch done!


Overall Progress: 100%|█████████▉| 248/249 [22:57<00:06,  6.01s/it]

✅ One batch done!


Overall Progress: 100%|██████████| 249/249 [23:12<00:00,  5.59s/it]

✅ One batch done!
✅ Processing complete! Results saved to: /content/drive/MyDrive/ev_detection_newresults4.parquet
